# 06 – Mô hình cây / boosting và phân tích lỗi

**Người thực hiện:** nhan (Mai Tấn Thiện Nhân)  
**Mục tiêu:** So sánh Decision Tree, Random Forest, XGBoost thật với nhiều tham số; phân tích lỗi mô hình cuối; tạo hình mẫu đầu vào → đầu ra cho slide.

**Quy tắc:** chạy TỪNG ô một, đổi tham số rồi chạy lại; mỗi ngày làm xong thì `git add` + `git commit -m "<mô tả đúng việc đã làm>"` + `git push`. Cuối buổi xuất HTML: `python ../scripts/export_notebook_html.py <notebook> nhan`.

**Các bước:**
1. `pip install xgboost` rồi chạy XGBoost thật (thay HistGradientBoosting giữa kỳ).
2. Decision Tree: quét max_depth.
3. Random Forest: quét n_estimators, max_features.
4. XGBoost: quét max_depth, learning_rate, n_estimators; thử số chiều SVD.
5. Phân tích lỗi mô hình Danh chọn: đọc 30 mẫu sai, phân loại nguyên nhân.
6. Hình mẫu đầu vào → đầu ra cho slide.

In [1]:
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.append('../src')
from run_experiment import run
from figlog import save_fig
from exp_logger import log_generic
from common import load_data, clean_title


## 1. Decision Tree – max_depth

In [4]:
res, _ = run(member='nhan', model='dt', features='svd', svd=150, params={'max_depth': None}, exp_name='dt_dNone', show_report=False)

[nhan_20261113_213047] dt/svd {'max_depth': None} split=strat
   train_time_s                 15.2813
   test_accuracy                0.9429
   test_precision_macro         0.9408
   test_recall_macro            0.9405
   test_f1_macro                0.9406
   test_f1_weighted             0.9429
   f1_Fridges                   0.8585
   f1_Freezers                  0.8371
   f1_Fridge_Freezers           0.9108
→ Đã ghi vào experiments/nhan/experiment_log.csv. Hãy mở file và điền cột nhan_xet.


> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/nhan/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: depth 20 được 0.932, depth 100 và None đều được 0.9406. Hôm nay dùng đặc trưng svd (150 chiều) chứ không phải word như hôm qua.
> - Nhận xét: để depth 20 thì F1 thấp hơn chút, tụt nhiều nhất ở Fridges với Freezers. Depth 100 với None ra y chang nhau vì em kiểm tra thì cây tự mọc chỉ sâu khoảng 37 tầng, nên giới hạn 100 không cắt gì hết. So với hôm qua (word) thì F1 tổng gần bằng nhưng mấy lớp tủ lạnh lại kém hơn.
> - Lần sau thử: chỉnh độ sâu không giúp được nhiều, em chuyển qua Random Forest xem ghép nhiều cây có tốt hơn không.

## 2. Random Forest

In [6]:
res, _ = run(member='nhan', model='rf', features='svd', params={'n_estimators': 300}, exp_name='rf_300', show_report=False)

[nhan_20261113_214408] rf/svd {'n_estimators': 300} split=strat
   train_time_s                 24.4265
   test_accuracy                0.9738
   test_precision_macro         0.9750
   test_recall_macro            0.9718
   test_f1_macro                0.9732
   test_f1_weighted             0.9738
   f1_Fridges                   0.9276
   f1_Freezers                  0.9299
   f1_Fridge_Freezers           0.9591
→ Đã ghi vào experiments/nhan/experiment_log.csv. Hãy mở file và điền cột nhan_xet.


> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/nhan/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: Random Forest 100 cây được 0.9716, 300 cây được 0.9732.
> - Nhận xét: ghép nhiều cây tốt hơn hẳn 1 cây (cây tốt nhất chỉ 0.9406, hơn khoảng 3 điểm). Mấy lớp tủ lạnh cũng lên rõ, Fridges với Freezers từ khoảng 0.84-0.86 lên khoảng 0.93. Tăng lên 300 cây thì chỉ nhích thêm chút xíu mà chạy lâu gấp đôi. Vẫn thua XGBoost hôm qua (0.9771) một chút.
> - Lần sau thử: không cần tăng số cây nữa, thử chỉnh max_features của Random Forest hoặc quay lại chỉnh XGBoost.

## 3. XGBoost thật

In [ ]:
res, _ = run(member='nhan', model='xgb', features='svd', svd=100,
             params={'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}, exp_name='xgb_base', show_report=False)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/nhan/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 4. Phân tích lỗi
Điền cấu hình cuối cùng mà nhóm chọn (hỏi Danh).

In [ ]:
from sklearn.model_selection import train_test_split
df = load_data(); df['clean'] = df.title.map(clean_title)
X, y = df[['clean','merchant_id']], df.category_label.values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
res, pipe = run(member='nhan', model='svc', features='full', params={'C': 1.0}, exp_name='error_analysis', show_report=False)
err = df.loc[Xte.index].assign(pred=pipe.predict(Xte))
err = err[err.category_label != err.pred]
err[['title','category_label','pred']].head(30)

**Tự phân loại 30 lỗi trên thành nhóm nguyên nhân** (chỉ có mã model / sản phẩm lai / tiếng Đức / nhãn sai…) và đếm:

| Nhóm nguyên nhân | Số mẫu | Ví dụ |
|---|---|---|
| … | … | … |

## 5. Hình mẫu đầu vào → đầu ra (cho slide)

In [ ]:
samples = Xte.sample(6, random_state=1)
pred = pipe.predict(samples)
fig, ax = plt.subplots(figsize=(10, 3.4)); ax.axis('off'); ax.set_ylim(0, 1)
for i, ((idx, r), p) in enumerate(zip(samples.iterrows(), pred)):
    true = df.loc[idx, 'category_label']; ok = '✓' if p == true else '✗'
    ax.text(0.0, 0.86 - i * 0.155, df.loc[idx, 'title'][:60], fontsize=10, family='monospace')
    ax.text(0.68, 0.86 - i * 0.155, f'→ {p}  {ok}', fontsize=10, color='green' if p == true else 'red')
ax.set_title('Đầu vào (tiêu đề sản phẩm) → Đầu ra (ngành dự đoán)')
save_fig(fig, 'nhan_sample_io', notebook='06_Nhan_Cay_Boosting_PhanTichLoi.ipynb', member='nhan', cell='Mục 5',
         y_nghia='(tự viết)')

## 6. Kết luận

**Xếp hạng (Macro-F1 trên tập test):**

1. XGBoost: 0.9771 (chạy hôm 12/11, svd 100 chiều, 200 cây)
2. Random Forest: 0.9732 (300 cây), 0.9716 (100 cây)
3. Decision Tree: 0.9406 (không giới hạn độ sâu), 0.9320 (depth 20)

**Vì sao ghép nhiều cây tốt hơn 1 cây?**

Một cây đơn rất dễ học thuộc tập train, chỉ cần đổi dữ liệu một chút là cây đổi hẳn, nên đoán trên tập test không ổn định. Em thử giới hạn độ sâu thì cũng không đỡ hơn, còn kém đi.

Random Forest thì cho nhiều cây học trên các phần dữ liệu khác nhau rồi bỏ phiếu, cây này sai thì cây khác bù lại, nên kết quả ổn định hơn. Tăng từ 100 lên 300 cây chỉ thêm chút xíu, chắc khoảng 100 cây là đủ.

XGBoost thì cây sau học sửa lỗi của cây trước nên cao nhất. Nhưng XGBoost em chạy với svd 100 chiều, còn hôm nay chạy svd 150 chiều, nên so sánh chưa thật công bằng. Mấy lớp tủ lạnh (Fridges, Freezers, Fridge Freezers) vẫn là chỗ dễ nhầm nhất ở cả 3 mô hình.